# From Gaussian beliefs to certified pdSTL lane execution

This notebook runs the **same lane-change and lane-merge pipeline as the project runners**. It is organized for a lab presentation: configuration → beliefs → physical predicates → pdSTL formula → receding-horizon optimization → sampled execution → certified interval and publication figures.

The two numbers that should not be confused are:

- **Certified hard pdSTL interval** $[L,U]$: the reported probabilistic satisfaction result.
- **Smooth score**: a differentiable surrogate used only by the optimizer.

Physical success is evaluated on the executed trajectory: complete the target-lane dwell in time, without collision or road violation; a merge must also complete before the ramp ends.

## 1. Setup

Run Jupyter from anywhere inside the repository. The small path block finds the repository root; all simulation behavior still comes from the project modules and editable YAML files.

In [ ]:
from pathlib import Path
from html import escape
import sys

import matplotlib.pyplot as plt
import torch
from IPython.display import HTML, Image, Markdown, display

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").is_file():
    raise RuntimeError("Open this notebook from inside the probabilistic-dstl repository.")
sys.path.insert(0, str(ROOT / "src"))

from planning.environment import lane_local_window, lane_subformulas
from planning.runners import _lane_initial_state, run_lane_change, setup_problem
from utils import get_device, load_config
from visualization.animation import animate_mpc
from visualization.planning import plot_lane_merge

DEVICE = get_device()
OUTPUT_DIR = ROOT / "outputs" / "lane_notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCENARIOS = {
    "Lane change": ROOT / "configs/scenarios/lane_change.yaml",
    "Lane merge": ROOT / "configs/scenarios/lane_merge.yaml",
}
MAKE_ANIMATIONS = True  # Set False for a faster rerun after planning.
print(f"Repository: {ROOT}")
print(f"Compute device: {DEVICE}")
print(f"Notebook outputs: {OUTPUT_DIR}")

## 2. Scenario configuration

The notebook does not hide or copy parameters. Change the normal files in `configs/scenarios/` and rerun. For traffic, `x0` is initial longitudinal position and `speed` is longitudinal speed; `x0_mean = [x, y, vx, vy]` is the ego initial mean.

In [ ]:
configs = {name: load_config(path) for name, path in SCENARIOS.items()}
rows = [
    r"| Scenario | $\Delta t$ | Horizon | Ego $[x,y,v_x,v_y]$ | Dwell window | Traffic $(x_0, v)$ |",
    "|---|---:|---:|---|---|---|",
]
for name, cfg in configs.items():
    traffic = ", ".join(
        f"{car['name']} ({car['x0']:g}, {car['speed']:g})"
        for car in cfg["traffic"]
    )
    window = cfg["task"]["start_window_seconds"]
    rows.append(
        f"| {name} | {cfg['dt']:g} s | {cfg['H']} | {cfg['x0_mean']} | "
        f"{window} s; dwell {cfg['task']['dwell_seconds']:g} s | {traffic} |"
    )
display(Markdown("\n".join(rows)))

## 3. Initial Gaussian beliefs

The planner does not optimize a single deterministic path. Each state is a Gaussian belief $(\mu_t, \Sigma_t)$. The ego and traffic covariances propagate through the dynamics, and collision predicates use their **relative Gaussian belief**. Vehicle footprint dimensions and configured safety margins define the collision envelope.

In [ ]:
problems = {}
initial_states = {}
for name, cfg in configs.items():
    problem = setup_problem(cfg, device=DEVICE, with_environment=True)
    state = _lane_initial_state(problem)
    problems[name] = problem
    initial_states[name] = state

    ego_std = torch.diag(state[1]).sqrt().detach().cpu().tolist()
    print(f"{name}")
    print(f"  ego mean       = {state[0].detach().cpu().tolist()}")
    print(f"  ego std        = {ego_std}")
    for car, mean in zip(cfg["traffic"], state[3]):
        print(f"  {car['name']:<12} = {mean.detach().cpu().tolist()}")
    print()

## 4. The lane pdSTL task

For every feasible dwell start $\tau$, the generated formula requires:

$$
\varphi_\tau = \mathbf{G}_{[0,\tau+d]}(\text{road} \land \text{collision-free})
\;\land\; \mathbf{G}_{[\tau,\tau+d]}(\text{target lane}).
$$

The merge additionally checks the ego front bumper at completion:

$$x_{\text{ego}} + \tfrac{1}{2}\ell_{\text{ego}} \leq x_{\text{ramp end}}.$$

The overall task is the disjunction over feasible starts, $\varphi=\bigvee_\tau\varphi_\tau$. This formula and the sampled `goal_reached` decision use the same derived time steps and geometry.

In [ ]:
for name, problem in problems.items():
    cfg = problem.cfg
    task = problem.env.metadata["task"]
    local_env = lane_local_window(
        problem.env, 0, initial_states[name][0], cfg, streak=0
    )
    pieces = lane_subformulas(local_env, cfg["H"])
    start, end = task["start_end_steps"]
    latest_start = end - task["dwell_steps"]
    margin = problem.env.metadata["safety_margin"]
    print(f"{name}")
    print(f"  dwell-start steps: {start}...{latest_start}")
    print(f"  completion deadline: step {end} ({end * cfg['dt']:g} s)")
    print(f"  dwell transitions: {task['dwell_steps']}")
    print(f"  safety margins: {margin}")
    print(f"  named components: {', '.join(pieces)}")
    display(HTML(
        f"<details><summary>Expand the exact generated formula</summary>"
        f"<pre style='white-space: pre-wrap'>{escape(str(pieces['overall']))}</pre></details>"
    ))
    print()

## 5. Receding-horizon planning and sampled execution

At each execution step, the real runner performs the following loop:

1. Propagate ego and traffic Gaussian beliefs over the planning horizon.
2. Optimize bounded longitudinal/lateral controls using the smooth pdSTL lower surrogate.
3. Evaluate the final plan with the **hard pdSTL interval semantics**.
4. Execute only the first control and sample the configured process noise.
5. Check mutually exclusive physical outcomes, then replan.

The following cell runs both complete configured scenarios. It can take a few minutes on CPU.

In [ ]:
results = {}
for name, path in SCENARIOS.items():
    print(f"\nRunning {name} from {path.relative_to(ROOT)}")
    results[name] = run_lane_change(
        str(path),
        show=False,
        save=False,
        live=False,
        live_optimization=False,
    )

## 6. Outcomes and final certified results

`success` below is the physical execution outcome. The final pdSTL result is only the hard interval from the last solved planning window. The smooth score is included to explain the optimizer, but is not called certified.

In [ ]:
summary = [
    "| Scenario | Outcome | Executed time | Planning windows | Certified hard interval | Smooth score |",
    "|---|---|---:|---:|---:|---:|",
]
for name, result in results.items():
    dt = configs[name]["dt"]
    elapsed = (len(result.states) - 1) * dt
    if result.window_plans:
        final = result.window_plans[-1]
        interval = f"[{final.hard_interval[0]:.4f}, {final.hard_interval[1]:.4f}]"
        smooth = f"{final.smooth_lower:.4f}"
    else:
        interval, smooth = "n/a", "n/a"
    summary.append(
        f"| {name} | **{result.stopped_reason.replace('_', ' ')}** | "
        f"{elapsed:.1f} s | {len(result.window_plans)} | {interval} | {smooth} |"
    )
display(Markdown("\n".join(summary)))

### Why can execution succeed when the smooth score looks lower?

They answer different questions. Physical success checks the realized sampled trajectory. The hard interval applies exact pdSTL/Fréchet operations to the predicted belief trajectory. The smooth score replaces min/max operations with differentiable approximations so gradients exist; it is deliberately conservative and is only an optimization signal.

## 7. Recompute the certificate directly from the final belief rollout

This cell reconstructs the same time-local formula for the last planning window, evaluates it on that plan's Gaussian belief trajectory, and verifies that it equals the stored hard interval. This is the direct link from **belief → pdSTL → reported certificate**.

In [ ]:
for name, result in results.items():
    if not result.window_plans:
        print(f"{name}: no solved planning window")
        continue
    index = len(result.window_plans) - 1
    state = result.states[index]
    problem = problems[name]
    local_env = lane_local_window(
        problem.env, index, state[0], problem.cfg, streak=state[2]
    )
    formula = local_env.get_specification(problem.cfg["H"])
    recomputed = formula.probability_interval(
        result.window_plans[index].rollout.belief_trajectory
    ).detach().cpu()
    stored = torch.tensor(result.window_plans[index].hard_interval)
    torch.testing.assert_close(recomputed, stored)
    print(
        f"{name}: [{recomputed[0]:.4f}, {recomputed[1]:.4f}] "
        "(recomputed interval matches stored result)"
    )

## 8. Publication figures

Each scenario produces the same two figure products as the normal runner:

- a local trajectory view with ego, red traffic, selected plans, one-second annotations, and final certified interval;
- a combined view with trajectory, longitudinal/lateral controls, hard interval, smooth score, and timing constraints.

In [ ]:
figure_sets = {}
for name, result in results.items():
    stem = name.lower().replace(" ", "_")
    figure_sets[name] = plot_lane_merge(
        result,
        problems[name].env,
        dt=configs[name]["dt"],
        save_path=OUTPUT_DIR / f"{stem}.png",
        show=False,
    )
    display(Markdown(f"### {name}: local trajectory"))
    display(figure_sets[name]["trajectory"][0])
    display(Markdown(f"### {name}: combined result"))
    display(figure_sets[name]["combined"][0])

## 9. Executed animations

Animations show the executed position, current receding-horizon plan, surrounding traffic, timing window, and hard interval as execution advances. Set `MAKE_ANIMATIONS = False` in the setup cell when only static lab-meeting figures are needed.

In [ ]:
if MAKE_ANIMATIONS:
    for name, result in results.items():
        stem = name.lower().replace(" ", "_")
        gif_path = OUTPUT_DIR / f"{stem}.gif"
        animate_mpc(
            result,
            problems[name].env,
            dt=configs[name]["dt"],
            filename=gif_path,
            lane=True,
            show=False,
        )
        display(Markdown(f"### {name} animation"))
        display(Image(filename=str(gif_path)))
else:
    print("Animation generation skipped.")

## Takeaway

The controller plans over Gaussian beliefs, optimizes a smooth pdSTL surrogate, reports a hard certified probability interval, executes one control at a time, and separately applies physical success/failure definitions to the sampled trajectory. Lane change and lane merge share this pipeline; the merge adds taper containment and front-bumper-before-ramp-end completion.